In [1]:
from pathlib import Path
import re
import json
import shutil

In [2]:
BASE = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA"
)

INPUT_ROOT = BASE / "Extracted"
CLEANED_OUTPUT = BASE / "Cleaned_Generative"

CLEANED_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"📂 Input root:      {INPUT_ROOT}")
print(f"📂 Cleaned output:  {CLEANED_OUTPUT}")

📂 Input root:      C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\Extracted
📂 Cleaned output:  C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\Cleaned_Generative


In [3]:
MARKER_PATTERN = re.compile(r"^\[(PAGE \d+|TABLE|/TABLE|META|/META)\]$")

def clean_for_generative(text: str) -> str:
    """
    Cleans extracted PDF text for RAG use.
    - Preserves [PAGE N], [TABLE], [/TABLE], [META], [/META] markers
    - Removes PDF artifacts: broken hyphenation, lone page numbers, URLs, emails
    - Normalizes whitespace without destroying sentence structure
    """
    if not text:
        return ""

    # --- Line-level cleaning (skip structural markers) ---
    lines = text.split("\n")
    cleaned_lines = []
    for line in lines:
        if MARKER_PATTERN.match(line.strip()):
            cleaned_lines.append(line)  # pass through untouched
        else:
            line = re.sub(r"http\S+|www\S+|https\S+", "", line)   # remove URLs
            line = re.sub(r"\S+@\S+", "", line)                    # remove emails
            line = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]", "", line)  # control chars
            line = re.sub(r"[ \t]+", " ", line)                    # normalize spaces
            cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # --- Document-level cleaning ---
    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)                          # fix hyphenated line breaks
    text = re.sub(r"\bPage\s+\d+\s+of\s+\d+\b", "", text, flags=re.IGNORECASE)  # remove "Page X of Y"
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)                             # remove lone page numbers
    text = re.sub(r"\n{3,}", "\n\n", text)                                  # collapse excess blank lines

    return text.strip()

In [4]:
def process_txt_file(file: Path, out_category: Path, transform_fn) -> int:
    """
    Reads a TXT file, applies cleaning, writes to output.
    Returns number of characters removed.
    """
    original = file.read_text(encoding="utf-8")
    cleaned = transform_fn(original)
    chars_removed = len(original) - len(cleaned)
    (out_category / file.name).write_text(cleaned, encoding="utf-8")
    print(f"  ✅ TXT: {file.name}  (chars removed: {chars_removed:,})")
    return chars_removed

In [5]:
def process_json_schema(file: Path, out_category: Path) -> None:
    """
    Validates JSON schema files and copies them untouched.
    Stamps each file/item with _meta so downstream pipeline
    knows it's a schema chunk, not a text chunk.
    """
    dest = out_category / file.name

    if dest.exists():
        print(f"  ⏭️  SKIPPED (already exists): {file.name}")
        return

    try:
        data = json.loads(file.read_text(encoding="utf-8"))

        meta = {
            "chunk_type": "schema",
            "source_file": file.name,
            "category": file.parent.name
        }

        if isinstance(data, dict):
            data["_meta"] = meta
            print(f"  📋 SCHEMA: {file.name}  (top-level keys: {[k for k in data if k != '_meta']})")

        elif isinstance(data, list):
            for item in data:
                if isinstance(item, dict):
                    item["_meta"] = meta
            print(f"  📋 SCHEMA: {file.name}  (list of {len(data)} items)")

        else:
            # Scalar — just copy as-is, nothing to annotate
            shutil.copy2(file, dest)
            print(f"  📋 SCHEMA COPIED (scalar): {file.name}")
            return

        dest.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")

    except json.JSONDecodeError as e:
        print(f"  ❌ Invalid JSON: {file.name} → {e}")
    except Exception as e:
        print(f"  ❌ Failed: {file.name} → {e}")

In [6]:
def process_folders(
    input_root: Path = INPUT_ROOT,
    output_root: Path = CLEANED_OUTPUT,
    transform_fn = clean_for_generative
) -> None:
    """
    Walks all category folders in input_root.
    - TXT files: cleaned and written to output
    - JSON files: validated, annotated with _meta, copied untouched (schema pass-through)
    """
    if not input_root.exists():
        print(f"❌ Input folder does not exist: {input_root}")
        return

    total_txt   = 0
    total_json  = 0
    total_chars = 0

    for category_dir in input_root.iterdir():
        if not category_dir.is_dir():
            continue

        out_category = output_root / category_dir.name
        out_category.mkdir(parents=True, exist_ok=True)

        print(f"\n📁 Category: {category_dir.name}")

        # --- TXT files ---
        txt_files = list(category_dir.glob("*.txt"))
        if not txt_files:
            print("   (No TXT files found)")
        else:
            for file in txt_files:
                chars_removed = process_txt_file(file, out_category, transform_fn)
                total_txt   += 1
                total_chars += chars_removed

        # --- JSON schema files ---
        json_files = list(category_dir.glob("*.json"))
        if not json_files:
            print("   (No JSON files found)")
        else:
            for file in json_files:
                process_json_schema(file, out_category)
                total_json += 1

    print(f"\n{'='*50}")
    print(f"📊 Summary:")
    print(f"   TXT files processed : {total_txt}")
    print(f"   JSON schemas copied : {total_json}")
    print(f"   Total chars removed : {total_chars:,}")
    print(f"{'='*50}")

In [ ]:
process_folders()


📁 Category: Cleaned_Generative
   (No TXT files found)
   (No JSON files found)

📁 Category: Database Tables
   (No TXT files found)
  📋 SCHEMA: ACT_PAR.JSON  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])


  📋 SCHEMA: ART_PAR.JSON  (top-level keys: ['table_name', 'description', 'columns'])
  📋 SCHEMA: CHG_DAT.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: CHL_DAT .json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: CHL_DAT.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: MIE_DAT.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: MIL_DAT.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: MVT_DAT.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: OPE_DAT.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: OPL_DAT.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: QUA_PAR.json  (top-level keys: ['table_name', 'description', 'primary_key', 'columns'])
  📋 SCHEMA: RE

: 